# Machine Learning for Omics Data

This notebook demonstrates machine learning workflows for omics data:
- Feature selection
- Model training and evaluation
- AutoML
- Model interpretability with SHAP

In [ ]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

# MOAS imports
from backend.ml.models import (
    RandomForestModel, XGBoostModel, LightGBMModel, SVMModel
)
from backend.ml.feature_selection import FeatureSelector
from backend.ml.training import ModelTrainer
from backend.ml.explainability import SHAPExplainer
from backend.ml.automl import AutoMLPipeline

plt.style.use('seaborn-v0_8-whitegrid')
print("Libraries loaded!")

## 1. Generate Sample Data

In [ ]:
# Generate classification dataset
np.random.seed(42)

n_samples = 200
n_features = 500
n_informative = 50

# Create features
X = np.random.randn(n_samples, n_features)

# Create binary labels
y = np.random.binomial(1, 0.5, n_samples)

# Add signal to informative features
for i in range(n_informative):
    X[y == 1, i] += np.random.uniform(0.5, 1.5)
    X[y == 0, i] -= np.random.uniform(0.5, 1.5)

# Create DataFrame
feature_names = [f"Feature_{i}" for i in range(n_features)]
X_df = pd.DataFrame(X, columns=feature_names)
y_series = pd.Series(y, name='label')

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_df, y_series, test_size=0.2, random_state=42, stratify=y_series
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"Class distribution: {np.bincount(y)}")

## 2. Feature Selection

In [ ]:
# Initialize feature selector
selector = FeatureSelector()

# Variance filter
variance_selected = selector.variance_filter(X_train, threshold=0.1)
print(f"Features after variance filter: {len(variance_selected)}")

In [ ]:
# Univariate selection
univariate_selected = selector.univariate_selection(
    X_train, y_train,
    method='f_classif',
    n_features=100
)
print(f"Top features from univariate selection: {len(univariate_selected)}")
print(f"Top 10: {univariate_selected[:10]}")

In [ ]:
# Embedded selection (Random Forest importance)
embedded_selected = selector.embedded_selection(
    X_train, y_train,
    method='random_forest',
    n_features=50
)
print(f"Features from embedded selection: {len(embedded_selected)}")

In [ ]:
# Visualize feature importance
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
top_20 = importance_df.head(20)
ax.barh(range(20), top_20['importance'], color='#3498db')
ax.set_yticks(range(20))
ax.set_yticklabels(top_20['feature'])
ax.invert_yaxis()
ax.set_xlabel('Importance')
ax.set_title('Top 20 Features by Random Forest Importance')
plt.tight_layout()
plt.show()

## 3. Model Training

In [ ]:
# Use selected features
selected_features = embedded_selected
X_train_selected = X_train[selected_features]
X_test_selected = X_test[selected_features]

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_selected)
X_test_scaled = scaler.transform(X_test_selected)

print(f"Training with {len(selected_features)} features")

In [ ]:
# Train multiple models
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(probability=True, random_state=42),
}

# Cross-validation
cv_results = {}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    scores = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='roc_auc')
    cv_results[name] = {
        'mean': scores.mean(),
        'std': scores.std(),
        'scores': scores
    }
    print(f"{name}: AUC = {scores.mean():.3f} (+/- {scores.std()*2:.3f})")

In [ ]:
# Visualize CV results
fig, ax = plt.subplots(figsize=(10, 6))

model_names = list(cv_results.keys())
means = [cv_results[m]['mean'] for m in model_names]
stds = [cv_results[m]['std'] for m in model_names]

bars = ax.bar(model_names, means, yerr=stds, capsize=5, color='#3498db', alpha=0.8)
ax.set_ylabel('ROC-AUC Score')
ax.set_title('Cross-Validation Performance Comparison')
ax.set_ylim(0.5, 1)

for bar, mean in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{mean:.3f}', ha='center')

plt.tight_layout()
plt.show()

## 4. Model Evaluation

In [ ]:
# Train best model on full training set
best_model_name = max(cv_results, key=lambda x: cv_results[x]['mean'])
print(f"Best model: {best_model_name}")

best_model = models[best_model_name]
best_model.fit(X_train_scaled, y_train)

# Predictions
y_pred = best_model.predict(X_test_scaled)
y_proba = best_model.predict_proba(X_test_scaled)[:, 1]

# Metrics
print(f"\nTest Set Performance:")
print(f"  Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(f"  Precision: {precision_score(y_test, y_pred):.3f}")
print(f"  Recall: {recall_score(y_test, y_pred):.3f}")
print(f"  F1 Score: {f1_score(y_test, y_pred):.3f}")
print(f"  ROC-AUC: {roc_auc_score(y_test, y_proba):.3f}")

In [ ]:
# ROC Curve and Confusion Matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
auc = roc_auc_score(y_test, y_proba)

axes[0].plot(fpr, tpr, color='#3498db', lw=2, label=f'ROC (AUC = {auc:.3f})')
axes[0].plot([0, 1], [0, 1], color='gray', linestyle='--')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve')
axes[0].legend()

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Negative', 'Positive'],
    yticklabels=['Negative', 'Positive'],
    ax=axes[1]
)
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')
axes[1].set_title('Confusion Matrix')

plt.tight_layout()
plt.show()

## 5. Model Interpretability with SHAP

In [ ]:
# SHAP analysis
import shap

# Create explainer
if hasattr(best_model, 'feature_importances_'):
    explainer = shap.TreeExplainer(best_model)
else:
    explainer = shap.KernelExplainer(best_model.predict_proba, X_train_scaled[:100])

# Calculate SHAP values
shap_values = explainer.shap_values(X_test_scaled)

# Handle multi-class output
if isinstance(shap_values, list):
    shap_values = shap_values[1]  # Use positive class

print(f"SHAP values calculated for {len(shap_values)} samples")

In [ ]:
# SHAP Summary Plot
plt.figure(figsize=(12, 8))
shap.summary_plot(
    shap_values, 
    X_test_selected,
    feature_names=selected_features,
    max_display=20,
    show=False
)
plt.title('SHAP Feature Importance')
plt.tight_layout()
plt.show()

In [ ]:
# SHAP Bar Plot (mean absolute values)
plt.figure(figsize=(10, 8))

mean_shap = np.abs(shap_values).mean(axis=0)
shap_importance = pd.DataFrame({
    'feature': selected_features,
    'importance': mean_shap
}).sort_values('importance', ascending=False)

top_20 = shap_importance.head(20)
plt.barh(range(20), top_20['importance'], color='#9b59b6')
plt.yticks(range(20), top_20['feature'])
plt.gca().invert_yaxis()
plt.xlabel('mean(|SHAP value|)')
plt.title('Top 20 Features by SHAP Importance')
plt.tight_layout()
plt.show()

In [ ]:
# Individual prediction explanation
sample_idx = 0

plt.figure(figsize=(14, 4))
shap.force_plot(
    explainer.expected_value[1] if isinstance(explainer.expected_value, list) else explainer.expected_value,
    shap_values[sample_idx],
    X_test_selected.iloc[sample_idx],
    matplotlib=True,
    show=False
)
plt.title(f'SHAP Explanation for Sample {sample_idx} (True: {y_test.iloc[sample_idx]}, Pred: {y_pred[sample_idx]})')
plt.tight_layout()
plt.show()

## 6. Save Results

In [ ]:
# Save model and results
# import joblib
# joblib.dump(best_model, 'models/best_model.joblib')
# joblib.dump(scaler, 'models/scaler.joblib')

# Save feature importance
# shap_importance.to_csv('results/shap_importance.csv', index=False)

# Save selected features
# pd.Series(selected_features).to_csv('results/selected_features.csv', index=False)

print("\nML Pipeline Complete!")
print(f"Best Model: {best_model_name}")
print(f"Test AUC: {roc_auc_score(y_test, y_proba):.3f}")
print(f"Features used: {len(selected_features)}")